# Douglas-fir Mortality Analysis

This notebook explores mortality trends in Douglas-fir forests using FIA data.

## Overview
- Analyze Douglas-fir mortality rates over time
- Examine geographic patterns by county
- Investigate size-class vulnerabilities
- Identify temporal trends and hotspots

## Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '../src')

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from fia_database import FIADatabaseConnector

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Imports successful!")

## Connect to Database

In [ ]:
# Update this path to your FIA database
db_path = "path/to/your/FIA_database.db"

# Initialize connector
connector = FIADatabaseConnector(db_path)
connector.connect()

print(f"Connected to database: {db_path}")

## Overall Douglas-fir Statistics

In [ ]:
# Get overall statistics
stats = connector.count_douglas_fir()

print("Douglas-fir Overall Statistics")
print("=" * 50)
for key, value in stats.items():
    print(f"{key:.<30} {value}")

# Store for visualization
total = stats['total_trees']
live = stats['live_trees']
dead = stats['dead_trees']
mortality_rate = stats['mortality_rate_pct']

## Visualization 1: Overall Mortality Breakdown

In [ ]:
# Create pie chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = ['#2ecc71', '#e74c3c']
sizes = [live, dead]
labels = [f'Live Trees\n({live:,})', f'Dead Trees\n({dead:,})']
explode = (0, 0.1)

ax1.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
        shadow=True, startangle=90, textprops={'fontsize': 11, 'weight': 'bold'})
ax1.set_title('Douglas-fir Mortality Status', fontsize=14, weight='bold')

# Bar chart
status_data = pd.DataFrame({
    'Status': ['Live', 'Dead'],
    'Count': [live, dead],
    'Color': colors
})

bars = ax2.bar(status_data['Status'], status_data['Count'], color=status_data['Color'], alpha=0.8, edgecolor='black')
ax2.set_ylabel('Number of Trees', fontsize=12, weight='bold')
ax2.set_title('Douglas-fir Tree Count by Status', fontsize=14, weight='bold')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}K'))

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('../output/mortality_breakdown.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Overall Mortality Rate: {mortality_rate}%")

## Mortality by Diameter Class

In [ ]:
# Get diameter class data
diameter_df = connector.get_mortality_by_diameter_class()
print(diameter_df)

## Visualization 2: Mortality by Diameter Class

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Mortality rate by diameter class
ax1.barh(diameter_df['diameter_class'], diameter_df['mortality_rate_pct'], 
         color='#e74c3c', alpha=0.8, edgecolor='black')
ax1.set_xlabel('Mortality Rate (%)', fontsize=12, weight='bold')
ax1.set_title('Douglas-fir Mortality Rate by Size Class', fontsize=14, weight='bold')
ax1.invert_yaxis()

# Add value labels
for i, v in enumerate(diameter_df['mortality_rate_pct']):
    ax1.text(v + 0.5, i, f'{v:.1f}%', va='center', fontweight='bold')

# Tree count by diameter class
ax2.bar(range(len(diameter_df)), diameter_df['tree_count'], 
        color='#3498db', alpha=0.8, edgecolor='black')
ax2.set_xticks(range(len(diameter_df)))
ax2.set_xticklabels(diameter_df['diameter_class'], rotation=45, ha='right')
ax2.set_ylabel('Number of Trees', fontsize=12, weight='bold')
ax2.set_title('Douglas-fir Distribution by Size Class', fontsize=14, weight='bold')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}K'))

plt.tight_layout()
plt.savefig('../output/mortality_by_diameter.png', dpi=300, bbox_inches='tight')
plt.show()

## Mortality by County

In [ ]:
# Get county-level data
county_df = connector.get_mortality_by_county()
print(f"Total counties: {len(county_df)}")
print("\nTop 10 counties by mortality rate:")
print(county_df.head(10).to_string(index=False))

## Visualization 3: Top Counties by Mortality Rate

In [ ]:
# Top 15 counties by mortality rate
top_counties = county_df.nlargest(15, 'mortality_rate_pct')

fig, ax = plt.subplots(figsize=(12, 8))

y_pos = np.arange(len(top_counties))
counties_labels = top_counties['county_name'] + ', ' + top_counties['state']

bars = ax.barh(y_pos, top_counties['mortality_rate_pct'], 
               color='#e74c3c', alpha=0.8, edgecolor='black')

ax.set_yticks(y_pos)
ax.set_yticklabels(counties_labels, fontsize=10)
ax.set_xlabel('Mortality Rate (%)', fontsize=12, weight='bold')
ax.set_title('Top 15 Counties: Douglas-fir Mortality Rate', fontsize=14, weight='bold')
ax.invert_yaxis()

# Add value labels
for i, (idx, row) in enumerate(top_counties.iterrows()):
    ax.text(row['mortality_rate_pct'] + 0.5, i, f"{row['mortality_rate_pct']:.1f}%", 
           va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('../output/top_counties_mortality.png', dpi=300, bbox_inches='tight')
plt.show()

## Visualization 4: Top Counties by Dead Tree Count

In [ ]:
# Top 15 counties by absolute dead tree count
top_dead = county_df.nlargest(15, 'dead_trees')

fig, ax = plt.subplots(figsize=(12, 8))

y_pos = np.arange(len(top_dead))
counties_labels = top_dead['county_name'] + ', ' + top_dead['state']

bars = ax.barh(y_pos, top_dead['dead_trees'], 
               color='#c0392b', alpha=0.8, edgecolor='black')

ax.set_yticks(y_pos)
ax.set_yticklabels(counties_labels, fontsize=10)
ax.set_xlabel('Number of Dead Trees', fontsize=12, weight='bold')
ax.set_title('Top 15 Counties: Douglas-fir Dead Tree Count', fontsize=14, weight='bold')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}K'))

# Add value labels
for i, (idx, row) in enumerate(top_dead.iterrows()):
    ax.text(row['dead_trees'] + 100, i, f"{int(row['dead_trees']):,}", 
           va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('../output/top_counties_dead_count.png', dpi=300, bbox_inches='tight')
plt.show()

## Temporal Trends by County

In [ ]:
# Query for temporal trends
temporal_query = """
SELECT 
    c.COUNTY as county_name,
    c.STABBR as state,
    s.INVYR as inventory_year,
    COUNT(t.TRE_CN) as tree_count,
    SUM(CASE WHEN t.STATUSCD = 1 THEN 1 ELSE 0 END) as live_trees,
    SUM(CASE WHEN t.STATUSCD = 2 THEN 1 ELSE 0 END) as dead_trees,
    ROUND(100.0 * SUM(CASE WHEN t.STATUSCD = 2 THEN 1 ELSE 0 END) / COUNT(t.TRE_CN), 2) as mortality_rate_pct
FROM TREE t
JOIN PLOT p ON t.PLT_CN = p.PLT_CN
JOIN SURVEY s ON p.SURVEYCD = s.SURVEYCD
JOIN COUNTY c ON p.COUNTYCD = c.COUNTYCD
WHERE t.SPCD = 202
GROUP BY c.COUNTY, c.STABBR, s.INVYR
ORDER BY c.COUNTY, s.INVYR
"""

temporal_df = connector.execute_query(temporal_query)
print(f"Temporal data retrieved: {len(temporal_df)} records")
print("\nSample data:")
print(temporal_df.head(10).to_string(index=False))

## Visualization 5: Mortality Trends Over Time

In [ ]:
if len(temporal_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Group by inventory year across all counties
    yearly_mortality = temporal_df.groupby('inventory_year').agg({
        'tree_count': 'sum',
        'dead_trees': 'sum',
        'live_trees': 'sum'
    }).reset_index()
    
    yearly_mortality['mortality_rate_pct'] = \
        100.0 * yearly_mortality['dead_trees'] / yearly_mortality['tree_count']
    
    ax.plot(yearly_mortality['inventory_year'], yearly_mortality['mortality_rate_pct'], 
           marker='o', linewidth=2.5, markersize=8, color='#e74c3c', label='Mortality Rate')
    
    ax.fill_between(yearly_mortality['inventory_year'], 
                    yearly_mortality['mortality_rate_pct'], 
                    alpha=0.2, color='#e74c3c')
    
    ax.set_xlabel('Inventory Year', fontsize=12, weight='bold')
    ax.set_ylabel('Mortality Rate (%)', fontsize=12, weight='bold')
    ax.set_title('Douglas-fir Mortality Trend Over Time', fontsize=14, weight='bold')
    ax.grid(True, alpha=0.3)
    
    # Add value labels
    for idx, row in yearly_mortality.iterrows():
        ax.text(row['inventory_year'], row['mortality_rate_pct'] + 0.3, 
               f"{row['mortality_rate_pct']:.1f}%", ha='center', fontweight='bold', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('../output/mortality_temporal_trend.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nYearly Mortality Trend:")
    print(yearly_mortality.to_string(index=False))
else:
    print("No temporal data available. This may require SURVEY table in your database.")

## Summary Statistics

In [ ]:
print("\n" + "="*70)
print("DOUGLAS-FIR MORTALITY ANALYSIS SUMMARY")
print("="*70)

print(f"\nOverall Statistics:")
print(f"  Total Douglas-fir trees: {total:,}")
print(f"  Live trees: {live:,} ({100*live/total:.1f}%)")
print(f"  Dead trees: {dead:,} ({mortality_rate}%)")

print(f"\nGeographic Coverage:")
print(f"  Number of counties: {len(county_df)}")
print(f"  Counties with 0% mortality: {len(county_df[county_df['mortality_rate_pct'] == 0])}")
print(f"  Counties with >50% mortality: {len(county_df[county_df['mortality_rate_pct'] > 50])}")

print(f"\nMortality Rate by County:")
print(f"  Minimum: {county_df['mortality_rate_pct'].min():.1f}%")
print(f"  Maximum: {county_df['mortality_rate_pct'].max():.1f}%")
print(f"  Mean: {county_df['mortality_rate_pct'].mean():.1f}%")
print(f"  Median: {county_df['mortality_rate_pct'].median():.1f}%")

print(f"\nSize Class Analysis:")
for idx, row in diameter_df.iterrows():
    print(f"  {row['diameter_class']:.<20} {row['mortality_rate_pct']:>6.1f}% mortality ({row['dead_trees']:,} dead)")

print("\n" + "="*70)

## Export Data

In [ ]:
# Create output directory if it doesn't exist
output_dir = Path('../output')
output_dir.mkdir(exist_ok=True)

# Export county data
county_df.to_csv(output_dir / 'douglas_fir_mortality_by_county.csv', index=False)
print("✓ Exported: douglas_fir_mortality_by_county.csv")

# Export diameter class data
diameter_df.to_csv(output_dir / 'douglas_fir_mortality_by_diameter.csv', index=False)
print("✓ Exported: douglas_fir_mortality_by_diameter.csv")

# Export temporal data if available
if len(temporal_df) > 0:
    temporal_df.to_csv(output_dir / 'douglas_fir_mortality_temporal.csv', index=False)
    print("✓ Exported: douglas_fir_mortality_temporal.csv")

print(f"\nAll files exported to: {output_dir.absolute()}")

## Cleanup

In [ ]:
# Close database connection
connector.disconnect()
print("Database connection closed.")